[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/certified-journeys/certified-journeys.github.io/blob/main/courses/metaflow-certified/notebooks/day-14-capstone-pipeline.ipynb#scrollTo=1a2b3c4d)

---
# Day 14 · Capstone: End-to-End ML Pipeline
**certified-journeys / metaflow-certified** · Day 14 · Exam

> **Goal for today:** Build and run a complete, production-grade ML pipeline — data ingest,
> feature engineering, hyperparameter search with `@foreach`, evaluation, model registration,
> and a rich `@card` summary — demonstrating mastery of every Metaflow concept from Days 1–13.


## Overview: What We're Building

This capstone pipeline has **five stages**, each implemented as one or more Metaflow steps:

```
start (ingest)
  └─ engineer_features
       └─ foreach: split_and_train  (one branch per hyperparameter config)
             └─ join: evaluate_all
                  └─ register_model
                       └─ end
```

**Decorators used per step:**

| Step | Decorators |
|------|------------|
| `start` | `@retry`, `@timeout`, `@card` |
| `engineer_features` | `@card` |
| `split_and_train` | `@retry`, `@timeout`, `@catch`, `@card` |
| `evaluate_all` | `@card` |
| `register_model` | `@card` |
| `end` | — |

**Key techniques:** `Parameter`, `@foreach`, `inputs` fan-in, artifact persistence,
Metaflow Client API, embedded matplotlib chart in card.

Run the cells in order. The full pipeline executes in under 60 seconds on a free Colab instance.


In [ ]:
%pip install -q metaflow scikit-learn pandas numpy matplotlib


## Part 1 · Pipeline Design and Parameters

Good pipeline design starts with **Parameters** — values injected at run time without
touching source code.

For this capstone we expose:
- `dataset` — which sklearn toy dataset to load (`iris` or `wine`)
- `test_size` — fraction of data held out for evaluation
- `random_seed` — ensures reproducible train/test splits

Hyperparameter configs are defined as a **Python list** and fanned out with `@foreach`.
Each branch receives one dict via `self.input` (Metaflow's magic attribute for foreach items).

```python
self.hp_configs = [
    {"C": 0.1,  "solver": "lbfgs"},
    {"C": 1.0,  "solver": "lbfgs"},
    {"C": 10.0, "solver": "lbfgs"},
    {"C": 1.0,  "solver": "saga"},
]
self.next(self.split_and_train, foreach="hp_configs")
```


In [ ]:
# Write the complete capstone pipeline to a .py file
# We build it section-by-section with comments so each concept is traceable

capstone_part1 = '''
# ============================================================
#  Capstone Pipeline  —  Metaflow for ML Engineers
#  Day 14: End-to-End ML Pipeline
# ============================================================
import io
import json

import matplotlib
matplotlib.use("Agg")   # headless rendering for Colab / CI
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from sklearn.datasets      import load_iris, load_wine
from sklearn.linear_model  import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics       import (
    accuracy_score, classification_report, confusion_matrix
)
from sklearn.preprocessing import StandardScaler

from metaflow import (
    FlowSpec, step, card, retry, timeout, catch, Parameter, current
)
from metaflow.cards import Markdown, Table, Image


# ── Helpers ────────────────────────────────────────────────────────────────

def make_confusion_matrix_image(cm, class_names):
    """Return PNG bytes for a confusion matrix heatmap."""
    fig, ax = plt.subplots(figsize=(5, 4))
    im = ax.imshow(cm, cmap="Blues")
    ax.set_xticks(range(len(class_names))); ax.set_xticklabels(class_names, rotation=45, ha="right")
    ax.set_yticks(range(len(class_names))); ax.set_yticklabels(class_names)
    plt.colorbar(im, ax=ax)
    for i in range(len(class_names)):
        for j in range(len(class_names)):
            ax.text(j, i, str(cm[i, j]), ha="center", va="center",
                    color="white" if cm[i, j] > cm.max() / 2 else "black")
    ax.set_xlabel("Predicted"); ax.set_ylabel("Actual")
    ax.set_title("Confusion Matrix")
    plt.tight_layout()
    buf = io.BytesIO()
    plt.savefig(buf, format="png", dpi=100)
    buf.seek(0)
    plt.close(fig)
    return buf.read()


def make_accuracy_bar_image(results):
    """Return PNG bytes for a bar chart of accuracy per config."""
    labels  = [r["label"] for r in results]
    accs    = [r["accuracy"] for r in results]
    colors  = ["#4CAF50" if a == max(accs) else "#90A4AE" for a in accs]
    fig, ax = plt.subplots(figsize=(7, 3))
    bars = ax.barh(labels, accs, color=colors)
    ax.set_xlim(0, 1.05)
    for bar, acc in zip(bars, accs):
        ax.text(acc + 0.01, bar.get_y() + bar.get_height() / 2,
                f"{acc:.4f}", va="center", fontsize=9)
    ax.set_xlabel("Accuracy")
    ax.set_title("Hyperparameter Search Results")
    plt.tight_layout()
    buf = io.BytesIO()
    plt.savefig(buf, format="png", dpi=100)
    buf.seek(0)
    plt.close(fig)
    return buf.read()

'''

print("Part 1 (imports + helpers) defined — length:", len(capstone_part1.splitlines()), "lines")


## Part 2 · Ingest and Feature Engineering Steps

The first two steps form the **data preparation stage**:

- **`start`** — loads the dataset from sklearn, stores raw arrays as artifacts, applies
  `@retry` + `@timeout` (as if this were a real network call), and generates a card
  summarising the dataset shape and class distribution.

- **`engineer_features`** — standardises features with `StandardScaler`, stores the
  scaled arrays and the fitted scaler (needed for inference), then fans out into
  `split_and_train` with four hyperparameter configs via `@foreach`.

**Artifact naming convention used throughout:**
- Raw data → `self.X_raw`, `self.y`, `self.feature_names`, `self.target_names`
- Scaled data → `self.X_scaled`, `self.scaler`
- Per-branch → `self.hp`, `self.model`, `self.accuracy`, `self.report_dict`
- Aggregated → `self.all_results`, `self.best_result`


In [ ]:
capstone_part2 = '''
class CapstoneMLPipeline(FlowSpec):
    """
    End-to-end ML pipeline demonstrating Metaflow mastery:
    ingest → features → foreach hyperparameter search → evaluate → register.
    """

    # ── Parameters ────────────────────────────────────────────────────────
    dataset     = Parameter("dataset",     default="iris",
                            help="Dataset to use: 'iris' or 'wine'")
    test_size   = Parameter("test_size",   default=0.2,
                            help="Fraction of data for test split (0.0–1.0)")
    random_seed = Parameter("random_seed", default=42,
                            help="Random seed for reproducibility")

    # ── Step 1: Ingest ────────────────────────────────────────────────────
    @retry(times=2, minutes_between_retries=0)  # safe for real data-fetch steps
    @timeout(seconds=60)                        # guard against hanging I/O
    @card                                       # generate an ingest report
    @step
    def start(self):
        """Load the dataset, store raw artifacts, generate a data profile card."""
        # ── Load dataset ───────────────────────────────────────────────
        if self.dataset == "wine":
            raw = load_wine(as_frame=True)
        else:
            raw = load_iris(as_frame=True)

        df = raw.frame
        self.feature_names = list(raw.feature_names)
        self.target_names  = list(raw.target_names)
        self.X_raw         = df[self.feature_names].values
        self.y             = df["target"].values
        self.n_classes     = len(self.target_names)
        self.n_samples     = len(self.y)
        self.n_features    = len(self.feature_names)

        # ── Class distribution ─────────────────────────────────────────
        unique, counts = np.unique(self.y, return_counts=True)
        self.class_counts = dict(zip([self.target_names[i] for i in unique], counts.tolist()))

        # ── Card: data profile ─────────────────────────────────────────
        current.card.append(Markdown(f"## Data Ingestion: `{self.dataset}` dataset"))
        summary_rows = [
            ["Property", "Value"],
            ["Dataset",   self.dataset],
            ["Samples",   str(self.n_samples)],
            ["Features",  str(self.n_features)],
            ["Classes",   str(self.n_classes)],
        ]
        current.card.append(Table(summary_rows))
        dist_rows = [["Class", "Count"]] + [[k, str(v)] for k, v in self.class_counts.items()]
        current.card.append(Markdown("### Class Distribution"))
        current.card.append(Table(dist_rows))

        self.next(self.engineer_features)

    # ── Step 2: Feature Engineering ───────────────────────────────────────
    @card
    @step
    def engineer_features(self):
        """Standardise features, define hyperparameter grid, fan out."""
        # ── StandardScaler: zero mean, unit variance ───────────────────
        self.scaler   = StandardScaler()
        self.X_scaled = self.scaler.fit_transform(self.X_raw)

        # ── Feature statistics card ────────────────────────────────────
        current.card.append(Markdown("## Feature Engineering"))
        stats_rows = [["Feature", "Mean (raw)", "Std (raw)", "Mean (scaled)", "Std (scaled)"]]
        for i, name in enumerate(self.feature_names):
            stats_rows.append([
                name,
                f"{self.X_raw[:, i].mean():.4f}",
                f"{self.X_raw[:, i].std():.4f}",
                f"{self.X_scaled[:, i].mean():.4f}",
                f"{self.X_scaled[:, i].std():.4f}",
            ])
        current.card.append(Table(stats_rows))
        current.card.append(Markdown(
            "Scaler fitted on full dataset and stored as `self.scaler` artifact "
            "for use at inference time."
        ))

        # ── Hyperparameter grid to fan out ─────────────────────────────
        self.hp_configs = [
            {"C": 0.1,  "solver": "lbfgs",  "max_iter": 1000},
            {"C": 1.0,  "solver": "lbfgs",  "max_iter": 1000},
            {"C": 10.0, "solver": "lbfgs",  "max_iter": 1000},
            {"C": 1.0,  "solver": "saga",   "max_iter": 2000},
        ]
        # foreach creates one parallel branch per config
        self.next(self.split_and_train, foreach="hp_configs")

'''

print("Part 2 (start + engineer_features) defined — length:",
      len(capstone_part2.splitlines()), "lines")


## Part 3 · Hyperparameter Search with `@foreach`

`@foreach` fans out into **N parallel branches**, one per item in the list.
Inside each branch `self.input` holds that branch's item from the list.

**Parallel execution model:**
```
engineer_features
  ├── split_and_train(hp={C:0.1,  solver:lbfgs})
  ├── split_and_train(hp={C:1.0,  solver:lbfgs})
  ├── split_and_train(hp={C:10.0, solver:lbfgs})
  └── split_and_train(hp={C:1.0,  solver:saga})
        └── evaluate_all  (join — all 4 branches merge here)
```

In local mode Metaflow runs branches sequentially. On AWS Batch / K8s they run
truly in parallel — zero code change needed.

**Fan-in with `inputs`:**
The join step receives `inputs` — a list of completed branches.
You collect artifacts by iterating: `[inp.accuracy for inp in inputs]`.


In [ ]:
capstone_part3 = '''
    # ── Step 3: Train (foreach branch) ────────────────────────────────────
    @timeout(seconds=120)                        # guard against runaway training
    @retry(times=1, minutes_between_retries=0)   # retry once on transient errors
    @catch(var="train_error")                    # catch so remaining branches keep running
    @card                                        # per-branch training card
    @step
    def split_and_train(self):
        """Train one LogisticRegression config; store model + metrics as artifacts."""
        self.hp = self.input   # hyperparameter dict for this branch

        # ── Train/test split (use same seed for fair comparison) ────────
        X_tr, X_te, y_tr, y_te = train_test_split(
            self.X_scaled, self.y,
            test_size=float(self.test_size),
            random_state=int(self.random_seed),
            stratify=self.y,           # preserve class ratios in both splits
        )

        # ── Train ───────────────────────────────────────────────────────
        self.model = LogisticRegression(
            C=self.hp["C"],
            solver=self.hp["solver"],
            max_iter=self.hp["max_iter"],
            multi_class="auto",
            random_state=int(self.random_seed),
        )
        self.model.fit(X_tr, y_tr)

        # ── Evaluate ────────────────────────────────────────────────────
        y_pred          = self.model.predict(X_te)
        self.accuracy   = float(accuracy_score(y_te, y_pred))
        self.report_str = classification_report(y_te, y_pred,
                                                target_names=self.target_names)
        self.report_dict = classification_report(y_te, y_pred,
                                                 target_names=self.target_names,
                                                 output_dict=True)
        self.cm         = confusion_matrix(y_te, y_pred)  # stored for the join step

        # ── Per-branch card ─────────────────────────────────────────────
        label = f"C={self.hp['C']} solver={self.hp['solver']}"
        current.card.append(Markdown(f"## Branch: {label}"))
        current.card.append(Markdown(f"**Accuracy: {self.accuracy:.4f}**"))

        # Confusion matrix image
        cm_img = make_confusion_matrix_image(self.cm, self.target_names)
        current.card.append(Image(cm_img, label=f"Confusion Matrix — {label}"))

        # Per-class metrics table
        rows = [["Class", "Precision", "Recall", "F1"]]
        for cls in self.target_names:
            m = self.report_dict.get(cls, {})
            rows.append([
                cls,
                f"{m.get('precision', 0):.4f}",
                f"{m.get('recall',    0):.4f}",
                f"{m.get('f1-score',  0):.4f}",
            ])
        current.card.append(Table(rows))

        self.next(self.evaluate_all)

'''

print("Part 3 (split_and_train) defined — length:",
      len(capstone_part3.splitlines()), "lines")


## Part 4 · Fan-in, Evaluation, and Model Registration

After all four branches complete, `evaluate_all` **joins** them:

```python
def evaluate_all(self, inputs):   # `inputs` is the list of branch tasks
    self.merge_artifacts(inputs, include=['X_scaled', 'y', ...])
    self.all_results = [inp.accuracy for inp in inputs]
```

`self.merge_artifacts(inputs)` copies all matching artifacts from branches into the join step.
You must call it before accessing any artifact that came from a branch.

**Model registration (local pattern):**
Production registration would push to MLflow, Weights & Biases, or SageMaker Model Registry.
Here we simulate it by writing a JSON manifest to disk and storing it as an artifact —
the pattern is identical regardless of backend.


In [ ]:
capstone_part4 = '''
    # ── Step 4: Evaluate All (join) ────────────────────────────────────────
    @card
    @step
    def evaluate_all(self, inputs):
        """
        Fan-in: collect results from all foreach branches,
        identify the best model, generate a comparison card.
        """
        # merge_artifacts brings branch artifacts into this step's namespace
        self.merge_artifacts(inputs, include=[
            "X_scaled", "y", "scaler", "feature_names", "target_names",
            "n_classes", "n_samples", "n_features", "class_counts",
            "dataset", "test_size", "random_seed",
        ])

        # ── Collect per-branch results ──────────────────────────────────
        self.all_results = []
        for inp in inputs:
            label = f"C={inp.hp['C']} solver={inp.hp['solver']}"
            self.all_results.append({
                "label":       label,
                "hp":          inp.hp,
                "accuracy":    inp.accuracy,
                "report_dict": inp.report_dict,
                "cm":          inp.cm.tolist(),  # ndarray → list for JSON-serialisable artifact
            })

        # ── Find best model ─────────────────────────────────────────────
        best_idx       = max(range(len(self.all_results)),
                             key=lambda i: self.all_results[i]["accuracy"])
        self.best_result = self.all_results[best_idx]
        # Store the winning model object (from the best branch's inp)
        self.best_model = list(inputs)[best_idx].model

        # ── Comparison card ─────────────────────────────────────────────
        current.card.append(Markdown("## Hyperparameter Search — All Results"))
        # Summary table
        rows = [["Config", "Accuracy", "Best?"]]
        for r in self.all_results:
            rows.append([
                r["label"],
                f"{r['accuracy']:.4f}",
                "✓ BEST" if r is self.best_result else "",
            ])
        current.card.append(Table(rows))

        # Accuracy bar chart
        bar_img = make_accuracy_bar_image(self.all_results)
        current.card.append(Image(bar_img, label="Accuracy by Config"))

        current.card.append(Markdown(
            f"**Winner:** `{self.best_result['label']}` "
            f"with accuracy **{self.best_result['accuracy']:.4f}**"
        ))
        self.next(self.register_model)

    # ── Step 5: Register Model ─────────────────────────────────────────────
    @card
    @step
    def register_model(self):
        """
        Simulate model registration: write a JSON manifest and store all
        reproducibility artifacts. In production replace the JSON write
        with an MLflow / SageMaker / W&B registry call.
        """
        import os, datetime

        # ── Build manifest ──────────────────────────────────────────────
        self.manifest = {
            "registered_at":  datetime.datetime.utcnow().isoformat() + "Z",
            "flow_run_id":    str(current.run_id),
            "dataset":        self.dataset,
            "test_size":      float(self.test_size),
            "random_seed":    int(self.random_seed),
            "best_hp":        self.best_result["hp"],
            "best_accuracy":  self.best_result["accuracy"],
            "feature_names":  self.feature_names,
            "target_names":   self.target_names,
            "model_class":    type(self.best_model).__name__,
        }

        # Write locally (production: push to registry)
        manifest_path = "/content/model_manifest.json"
        with open(manifest_path, "w") as f:
            json.dump(self.manifest, f, indent=2)

        # ── Registration card ────────────────────────────────────────────
        current.card.append(Markdown("## Model Registration"))
        current.card.append(Markdown(
            "> In production this step would call MLflow `mlflow.register_model()`, "
            "SageMaker `create_model()`, or W&B `run.log_artifact()`."
        ))
        manifest_rows = [["Field", "Value"]] + [
            [k, str(v)] for k, v in self.manifest.items()
        ]
        current.card.append(Table(manifest_rows))
        current.card.append(Markdown(
            f"Manifest written to `{manifest_path}`. "
            "All artifacts (scaler, model, manifest) stored as Metaflow artifacts for "
            "reproducibility — accessible via the Client API from any machine."
        ))

        self.next(self.end)

    # ── Step 6: End ────────────────────────────────────────────────────────
    @step
    def end(self):
        """Print a final summary — all artifacts available for downstream use."""
        print("=" * 60)
        print("CAPSTONE PIPELINE COMPLETE")
        print("=" * 60)
        print(f"Dataset       : {self.dataset}")
        print(f"Samples       : {self.n_samples}  Features: {self.n_features}")
        print(f"Configs tried : {len(self.all_results)}")
        print(f"Best config   : {self.best_result['label']}")
        print(f"Best accuracy : {self.best_result['accuracy']:.4f}")
        print(f"Run ID        : {current.run_id}")
        print("Artifacts stored: best_model, scaler, manifest, all_results")
        print("=" * 60)


if __name__ == "__main__":
    CapstoneMLPipeline()
'''

print("Part 4 (evaluate_all + register_model + end) defined — length:",
      len(capstone_part4.splitlines()), "lines")


## Part 5 · Assemble and Run the Complete Pipeline

Now we concatenate all four parts into a single `.py` file and run it end-to-end.

Expected output:
- 4 branches of `split_and_train` complete (each with its own card)
- `evaluate_all` picks the best config
- `register_model` writes `model_manifest.json`
- `end` prints a summary table
- Total wall-clock time: ~20–40 seconds on Colab free tier


In [ ]:
# Assemble the complete pipeline file
full_pipeline = capstone_part1 + capstone_part2 + capstone_part3 + capstone_part4

with open('/content/capstone_pipeline.py', 'w') as f:
    f.write(full_pipeline)

total_lines = len(full_pipeline.splitlines())
print(f"capstone_pipeline.py written — {total_lines} lines total")

# Quick syntax check before running
import ast
try:
    ast.parse(full_pipeline)
    print("Syntax check: PASSED")
except SyntaxError as e:
    print(f"Syntax error: {e}")


In [ ]:
# ── Run the capstone pipeline with the Iris dataset ────────────────────────
!python /content/capstone_pipeline.py run \
    --dataset iris \
    --test_size 0.2 \
    --random_seed 42


In [ ]:
# ── Re-run with the Wine dataset to verify parameterization ───────────────
!python /content/capstone_pipeline.py run \
    --dataset wine \
    --test_size 0.25 \
    --random_seed 99


## Part 6 · Verify Outputs with the Client API

After any Metaflow run you can programmatically retrieve every artifact using the Client API.
This is how CI pipelines, dashboards, and downstream services consume results — no shell
access needed, just a Python environment with `metaflow` installed.

Pattern:
```python
from metaflow import Flow
run   = Flow('CapstoneMLPipeline').latest_successful_run
model = run['end'].task.data.best_model        # the sklearn model object
scaler = run['end'].task.data.scaler            # the fitted StandardScaler
manifest = run['end'].task.data.manifest        # registration manifest dict
```


In [ ]:
# Retrieve artifacts from the latest successful run and run inference
from metaflow import Flow
import numpy as np

try:
    flow = Flow('CapstoneMLPipeline')
    run  = flow.latest_successful_run
    print(f"Run ID       : {run.id}")
    print(f"Successful   : {run.successful}")
    print(f"Steps        : {[s.id for s in run]}")

    # Load artifacts from the 'end' step
    end_task = list(run['end'])[0]
    best_model   = end_task.data.best_model
    scaler       = end_task.data.scaler
    manifest     = end_task.data.manifest
    best_result  = end_task.data.best_result
    feature_names = end_task.data.feature_names
    target_names  = end_task.data.target_names

    print(f"\nBest config  : {manifest['best_hp']}")
    print(f"Best accuracy: {manifest['best_accuracy']:.4f}")
    print(f"Model class  : {manifest['model_class']}")
    print(f"Registered at: {manifest['registered_at']}")

    # ── Live inference with retrieved model ─────────────────────────────
    # Create a synthetic sample (using dataset feature ranges)
    sample_raw = np.array([[5.0, 3.5, 1.4, 0.2]])  # iris-like: sepal/petal dims
    sample_scaled = scaler.transform(sample_raw)
    prediction = best_model.predict(sample_scaled)
    proba       = best_model.predict_proba(sample_scaled)

    print(f"\nSample features: {sample_raw[0].tolist()}")
    print(f"Predicted class : {target_names[prediction[0]]}")
    print(f"Class probabilities:")
    for name, prob in zip(target_names, proba[0]):
        print(f"  {name:20s}: {prob:.4f}")

except Exception as e:
    print(f"Note: {e}")
    print("Run the pipeline cells above first, then re-execute this cell.")


### What just happened?

- `Flow('CapstoneMLPipeline').latest_successful_run` returns the last run that reached `end` — **not** just the last run (which may have failed).
- `best_model` and `scaler` were loaded from the Metaflow datastore as full Python objects — no pickle/unpickle code needed; Metaflow handles serialisation.
- **The inference loop is fully reproducible**: same model, same scaler, same results — regardless of when or where you run it.
- In production `sample_raw` would come from a feature store or real-time request; everything else stays identical.
- `manifest['registered_at']` gives you a full audit trail: when, which run, which hyperparameters, which dataset version.


## Part 7 · Production Checklist

Before handing this pipeline to a new team, verify each item:

| Check | Why it matters |
|-------|---------------|
| `@retry` on all external I/O steps | Network blips kill pipelines; retries make them self-healing |
| `@timeout` guards | Prevents zombie steps from holding cluster resources forever |
| `@catch` on foreach branches | One bad branch should not kill the entire fan-out |
| `@card` on every major step | Observability without code access; stakeholders can review results |
| Parameters for dataset / seed / split | Reproducibility and easy re-runs with different data |
| `self.merge_artifacts` before accessing branch data | Required at every join step; skipping it causes AttributeError |
| Fitted scaler stored as artifact | Prevents train/serve skew — same scaler at inference as training |
| Manifest JSON with run_id | Audit trail: every deployed model traces back to an exact run |
| `.ndarray.tolist()` before storing in manifest | JSON is not ndarray-aware; always convert before serialising |
| Re-run with different dataset parameter | Verifies parameterisation actually works end-to-end |

> **Rule of thumb:** if you can't answer "which run produced this model in production?",
> your pipeline is not production-grade yet.


## Challenge


In [ ]:
# Challenge: Extend the capstone pipeline with two improvements:
#
# 1. Add a RandomForestClassifier to the hyperparameter grid (alongside LogisticRegression)
#    Hint: store the model class name in the hp dict and branch on it in split_and_train
#
# 2. Add a @card to the `end` step that shows:
#    - A Markdown summary of the entire run (dataset, n_samples, best config)
#    - The final accuracy bar chart (reuse make_accuracy_bar_image)
#    - A Table of all results sorted by accuracy (best first)
#
# Scaffold — copy capstone_pipeline.py and modify these sections:

challenge_scaffold = '''
# In engineer_features — extend hp_configs:
self.hp_configs = [
    # existing LR configs ...
    {"model": "lr",  "C": 1.0,   "solver": "lbfgs", "max_iter": 1000},
    # TODO: add RandomForest configs, e.g.:
    # {"model": "rf",  "n_estimators": 100, "max_depth": None},
    # {"model": "rf",  "n_estimators": 50,  "max_depth": 5},
]

# In split_and_train — branch on model type:
if self.hp.get("model", "lr") == "rf":
    from sklearn.ensemble import RandomForestClassifier
    self.model = RandomForestClassifier(
        # TODO: use params from self.hp
    )
else:
    self.model = LogisticRegression(
        C=self.hp["C"], solver=self.hp["solver"],
        max_iter=self.hp["max_iter"], random_state=int(self.random_seed)
    )

# In end — add @card and build the summary:
@card
@step
def end(self):
    # TODO: append Markdown summary, accuracy bar chart, sorted results Table
    pass
'''
print(challenge_scaffold)
# Write your solution, save it as /content/capstone_extended.py, and run it!


---
## Day 14 key concepts recap

| Concept | What to remember |
|---|---|
| `Parameter` | Injects values at run time — no source edits needed between runs |
| `@foreach` + `self.input` | Fan-out over a list; each branch gets its item via `self.input` |
| `merge_artifacts(inputs)` | Always call at the join step before reading any branch artifact |
| Decorator stack | `@timeout → @retry → @catch → @card → @step` |
| Storing sklearn objects | Metaflow serialises Python objects automatically — no pickle code |
| Fitted scaler as artifact | Critical for train/serve consistency; retrieve with Client API |
| `latest_successful_run` | Use this (not `latest_run`) to avoid loading a failed run's artifacts |
| Registration manifest | A JSON dict with run_id + hyperparams = full audit trail |
| `ndarray.tolist()` | Convert numpy arrays before storing in JSON-serialisable dicts |
| `matplotlib.use('Agg')` | Required for headless rendering in Colab / CI |

> **Tip:** Your capstone is proof you can build production-grade ML — structure it as if
> handing it off to a new team. Every decision (retry counts, timeout values, artifact names)
> should be self-documenting.

---
## Congratulations — You've Completed Metaflow for ML Engineers!

You have covered:

- **DAG fundamentals:** `FlowSpec`, `@step`, `self.next`, `@foreach`, join
- **Reliability:** `@retry`, `@timeout`, `@catch`
- **Infrastructure:** `@resources`, `@conda`
- **Observability:** `@card`, component API, Metaflow Client API
- **Production patterns:** Parameters, artifact versioning, model registration

Mark Day 14 complete in your [tracker](../index.html).
